In [ ]:
# Exercice 1 — Annoter et passer mypy strict (≈ 30 min)
# Reprendre un projet précédent (par exemple le système de paiements du M4) et :

# Ajouter les annotations de type sur toutes les fonctions et méthodes.
# Exécuter mypy --strict module.py.
# Corriger les erreurs jusqu'à 0 erreur.

from dataclasses import dataclass, field, replace
from abc import ABC, abstractmethod

@dataclass(frozen=True)
class Money:
    amount: int = 0 # ? in cents
    currency: str = "EUR"

    def __add__(self, other: "Money") -> "Money":
        return replace(self, amount = self.amount + other.amount)

    def __mul__(self, other: int) -> "Money":
        return replace(self, amount = self.amount * other)

@dataclass
class PaymentMethod(ABC):
    @abstractmethod
    def charge(self, amount: Money) -> str:
        ...

    @abstractmethod
    def refund(self, transaction_id: str) -> bool:
        ...

@dataclass
class CreditCardPayment(PaymentMethod):
    card: str

    def charge(self, amount: Money) -> str:
        # ? fake implementation
        return str(amount)
    def refund(self, transaction_id: str) -> bool:
        # ? fake implementation
        return transaction_id != ""

@dataclass
class BankTransferPayment(PaymentMethod):
    pass

@dataclass
class WalletPayment(PaymentMethod):
    pass


@dataclass
class OrderItem:
    sku: str
    quantity: int
    unit_price: Money

@dataclass
class Order:
    id: str
    items: list[OrderItem]
    method: PaymentMethod
    total: Money = field(default=Money())

    def __post_init__(self) -> None:
        for i in items:
            self.total += i.unit_price * i.quantity

items = [OrderItem(sku="A", quantity=2, unit_price=Money(500, "EUR"))]
order = Order(id="O-1", items=items, method=CreditCardPayment(card="4242..."))
print(order.total)                  # Money(amount=1000, currency='EUR')

tx_id = order.method.charge(order.total)
assert order.method.refund(tx_id) is True

Money(amount=500, currency='EUR')


In [ ]:
# Exercice 2 — Générique typé (≈ 30 min)

# Écrire une classe Stack[T] (pile générique) avec :

# push(item: T) -> None
# pop() -> T
# peek() -> T
# __len__() -> int

# Vérifier en mypy strict que Stack[int] et Stack[str] produisent des types corrects,
#   et qu'un push(str) sur une Stack[int] est signalé comme erreur.

# Bonus : utiliser la syntaxe Python 3.12+ (class Stack[T]:).

class Stack[T]:
    def __init__(self) -> None:
        self._items: list[T] = []

    def push(self, item: T) -> None:
        self._items.append(item)

    def pop(self) -> T:
        return self._items.pop()

    def peek(self) -> T:
        return self._items[-1]

    def __len__(self) -> int:
        return len(self._items)

example_int_stack = Stack[int]()
example_int_stack.push(1)
example_int_stack.push(2)
print(example_int_stack.pop())  # 2

example_str_stack = Stack[str]()
example_str_stack.push("hello")
example_str_stack.push("world")
print(example_str_stack.pop())  # "world"

example_int_stack.push("string")  # raise

In [ ]:
# Exercice 3 — Découvrir une version (≈ 20 min)
# Choisir une version Python parmi 3.10, 3.11 ou 3.12 et écrire 3 mini-snippets démontrant 3 nouveautés de cette version :

# 3.10 : match-case, parenthesized context managers, meilleures erreurs.
# 3.11 : Self type, ExceptionGroup, tomllib.
# 3.12 : syntaxe générique PEP 695, f-string améliorée, per-interpreter GIL.
# Rédiger pour chaque snippet une explication d'1-2 lignes du gain par rapport à la version précédente.

def match_case_example(value: int) -> str:
    match value:
        case 1:
            return "One"
        case 2:
            return "Two"
        case _:
            return "Other"
# plus lisible et extensible que if-elif-else.

def parenthesized_context_manager_example() -> None:
    with (
        open("file1.txt") as _f1,
        open("file2.txt") as _f2
    ):
        pass
# permet de grouper plusieurs context managers sur plusieurs lignes, améliorant la lisibilité.

def better_error_example() -> None:
    try:
        _result = 1 / 0
    except ZeroDivisionError as e:
        raise ValueError("Invalid operation") from e
# fournit des messages d'erreur plus clairs et plus utiles pour le débogage.

better_error_example()

In [ ]:
# Exercice 4 — Empaqueter un module (≈ 45 min)
# Créer un projet hello_noledj/ avec la structure recommandée (src/, tests/, pyproject.toml).

# Une fonction hello_noledj.greet(name: str) -> str qui renvoie "Hello, {name}!".
# Un test pytest qui valide le comportement.
# Un pyproject.toml minimal avec [build-system] et [project].
# Construire le wheel : python -m build.
# Installer le wheel dans un environnement virtuel propre et vérifier import hello_noledj.
# Bonus : publier sur TestPyPI et installer depuis là.

def greet(name: str) -> str:
    return f"Hello, {name}!"

def test_greet():
    assert greet("Alice") == "Hello, Alice!"
    assert greet("Bob") == "Hello, Bob!"
    assert greet("") == "Hello, !"

In [ ]:
# Exercice 5 — Investiguer __pycache__ (≈ 15 min)
# Créer un module m.py avec une fonction simple.
# L'importer depuis un autre script → observer le __pycache__/m.cpython-XYZ.pyc créé.
# Modifier m.py → ré-importer → constater que m.cpython-XYZ.pyc est régénéré.
# Lancer python -m dis __pycache__/m.cpython-XYZ.pyc (ou python -c "import dis; dis.dis(__import__('m'))") → observer le bytecode.
# Documenter ce qu'on apprend en 5 à 10 lignes.

# *****************************
# __pycache__/ est un dossier qui stocke les fichiers de bytecode Python (.pyc) pour accélérer le chargement des modules.
# À l'import d'un module, Python compile le code source en bytecode
#   et le stocke dans __pycache__/ pour une exécution plus rapide lors des imports ultérieurs.
# Si le fichier source est modifié, Python régénère automatiquement son .pyc.
# Le bytecode est une optimisation du code source pour l'interpréteur Python.
# Le module dis sert à examiner le bytecode et comprendre comment Python exécute le code.
#   Cela permet de mieux comprendre le fonctionnement interne de Python et d'optimiser les performances du code.

In [ ]:
# 6. Mini-défi de synthèse — la micro-bibliothèque (≈ 1 à 2 jours, mini-projet final du parcours Python)
# C'est le mini-projet annoncé dans le parcours global Python. Il rassemble les modules M2 à M7.

# Objectif. Concevoir, packager et publier une micro-bibliothèque Python qui démontre la maîtrise des concepts du parcours.

# ? Spécifications

# Domaine au choix : par exemple, une bibliothèque de manipulation de devises (@dataclass(frozen=True) Money),
#   un client API minimaliste, un parseur de fichiers .ini, un mini-ORM SQLite, etc.
# * Modélisation : utilise @dataclass, héritage propre, MRO maîtrisé si héritage multiple, dunders implémentés (M2-M4).
# * Concurrence : au moins une fonction qui exploite threading ou multiprocessing (M5).
# * Décorateurs : au moins un décorateur custom (timer, cache, retry, validation — M6).
# * Typing : annotations partout, mypy --strict passe sans erreur (M7).
# * Packaging : pyproject.toml propre, construction python -m build, wheel installable.
# * Qualité : ruff check et ruff format passent, tests pytest avec couverture > 80 %.
# * Documentation : README.md avec usage, API, et exemple d'installation.
# * Publication : déposé sur TestPyPI au minimum.

# ! Critères de validation

# [x] mypy --strict : 0 erreur.
# [x] ruff check . : 0 erreur.
# [x] pytest --cov : ≥ 80 %.
# [x] python -m build : produit un wheel et une sdist.
# [x] pip install --index-url https://test.pypi.org/simple/ <package> fonctionne dans un environnement propre.
# [x] Lecture rapide du code par un pair : noms clairs, structure cohérente, pas de TODO en attente.

from abc import ABC, abstractmethod
from concurrent.futures import ProcessPoolExecutor
from dataclasses import dataclass, field, replace
from functools import wraps
from math import log10, floor
from time import perf_counter
from typing import Any, Callable


def round_significant(x: float, sig: int = 3) -> float:
    if x == 0:
        return 0
    else:
        return round(x, sig - int(floor(log10(abs(x)))) - 1)


def timer(func: Callable[..., Any]) -> Callable[..., Any]:
    @wraps(func)
    def wrapper(*args: Any, **kwargs: Any) -> Any:
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = round_significant(perf_counter() - start)
        print(f"{func.__name__} took {elapsed}s")
        return result

    return wrapper


@dataclass(frozen=True)
class Money:
    amount: int = 0  # ? in cents
    currency: str = "EUR"

    def __add__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("Cannot add Money with different currencies")
        return replace(self, amount=self.amount + other.amount)

    def __sub__(self, other: "Money") -> "Money":
        if self.currency != other.currency:
            raise ValueError("Cannot subtract Money with different currencies")
        return replace(self, amount=self.amount - other.amount)

    def __mul__(self, other: int) -> "Money":
        return replace(self, amount=self.amount * other)

    def __rmul__(self, other: int) -> "Money":
        return self.__mul__(other)

    def __str__(self) -> str:
        return f"{self.amount / 100:.2f} {self.currency}"


@dataclass
class PaymentMethod(ABC):
    @abstractmethod
    def charge(self, amount: Money) -> str: ...

    @abstractmethod
    def refund(self, transaction_id: str) -> bool: ...


@dataclass
class CreditCardPayment(PaymentMethod):
    card: str

    def charge(self, amount: Money) -> str:
        # ? fake implementation
        return str(amount)

    def refund(self, transaction_id: str) -> bool:
        # ? fake implementation
        return transaction_id != ""


@dataclass
class BankTransferPayment(PaymentMethod):
    pass


@dataclass
class WalletPayment(PaymentMethod):
    pass


@dataclass
class OrderItem:
    sku: str
    quantity: int
    unit_price: Money

    def __init__(self, sku: str, quantity: int, unit_price: Money) -> None:
        if quantity < 0:
            raise ValueError("Quantity cannot be negative")
        if unit_price.amount <= 0.0:
            raise ValueError("Unit price cannot be zero or negative")
        self.sku = sku
        self.quantity = quantity
        self.unit_price = unit_price


@dataclass
class Order:
    id: str
    items: list[OrderItem]
    method: PaymentMethod
    total: Money = field(init=False, default=Money())

    def __post_init__(self) -> None:
        for i in self.items:
            self.total += i.unit_price * i.quantity


def cache_fib(func: Callable[[int], int]) -> Callable[[int], int]:
    cache_dict: dict[int, int] = {}

    def wrapper(n: int) -> int:
        if n not in cache_dict:
            cache_dict[n] = func(n)
        return cache_dict[n]

    return wrapper


@cache_fib
def fib(n: int) -> int:
    if n < 0:
        raise ValueError("Negative arguments are not supported")
    if n <= 1:
        return n
    else:
        return fib(n - 1) + fib(n - 2)


def uncached_fib(n: int) -> int:
    if n < 0:
        raise ValueError("Negative arguments are not supported")
    if n <= 1:
        return n
    else:
        return uncached_fib(n - 1) + uncached_fib(n - 2)


@timer
def run_cached(item_range: int = 36) -> Order:
    items = [
        OrderItem(sku="M169-wristwatch", quantity=fib(n), unit_price=Money(77, "EUR"))
        for n in range(item_range)
    ]
    order = Order(id="O-1", items=items, method=CreditCardPayment(card="4242..."))
    print("Calculated order total:", order.total)
    return order


@timer
def run_uncached(item_range: int = 36) -> Order:
    with ProcessPoolExecutor(max_workers=4) as pool:
        quantities = list(pool.map(uncached_fib, range(item_range)))

    items = [
        OrderItem(
            sku="M169-wristwatch", quantity=quantities[n], unit_price=Money(77, "EUR")
        )
        for n in range(item_range)
    ]
    order = Order(id="O-1", items=items, method=CreditCardPayment(card="4242..."))
    print("Calculated order total:", order.total)
    return order

if __name__ == "__main__":
    result_cached = run_cached()
    result_uncached = run_uncached()

    tx_id = result_cached.method.charge(result_cached.total)
    assert result_cached.method.refund(tx_id) is True

Le module M7 est validé lorsque :

- [x] L'apprenant peut annoter un prototype de fonction (paramètres, retour, optionnel).
- [x] Il a fait passer un module en mypy --strict sans erreur.
- [x] Il peut utiliser TypeVar ou la syntaxe générique 3.12+ pour une classe paramétrée.
- [x] Il peut citer 4 différences Py2 / Py3 et 3 nouveautés post-3.7.
- [x] Il sait écrire un pyproject.toml minimal et construire un wheel.
- [x] Il a publié au moins un paquet sur TestPyPI.
- [x] Il explique ce qu'est un .pyc et où il vit.
- [x] Le mini-projet de synthèse passe tous les critères de validation.